# Spotify2YouTube
> di Mario Gabriele Carofano

Notebook interattivo per importare playlist da Spotify, cercare i relativi brani su YouTube e scaricarne automaticamente i file multimediali (audio o video).

### Funzionalità principali
1.	Estrazione Playlist da Spotify:
	-	Autenticazione tramite le API Spotify (spotipy).
	-	Recupera titolo e artisti di tutti i brani di una playlist.
	-	Esporta i dati in un file CSV locale.
	-	Il percorso dell'output può essere modificato.
2.	Ricerca su YouTube:
	-	Utilizza yt-dlp per cercare il miglior video corrispondente ad ogni brano.
	-	Supporta molteplici modalità di ricerca definite da 'QueryType'.
3.	Download automatico:
	-	Legge i file CSV generati e utilizza 'yt-dlp' per scaricare i file multimediali (audio o video).
	-	Il percorso dei download può essere modificato.

### Istruzioni per l'uso
Dal menù interattivo, si può collegare l’account Spotify Developer digitando il client ID, il client SECRET e il redirect URI (oppure si possono salvare in un file config.py dedicato).

Esegui le celle in ordine. Le celle 5-9 possono essere rieseguite più volte in base alle operazioni che vuoi compiere.

---

## Setup e Importazione Librerie

Importa le librerie necessarie per l'esecuzione del notebook, già incluse in `utilities.py` e `functions.py`, oltre a tutte le funzioni custom.

In [ ]:
# Funzioni e utility custom del progetto
from functions import *
from utilities import *

print("✅ Librerie importate correttamente.")

## Configurazione e autenticazione Spotify / Youtube

Carica le credenziali Spotify tramite `load_spotify_config()`. Se non sono presenti, richiede manualmente `client_id`, `client_secret` e `redirect_uri` all'utente.

Successivamente, vengono creati i client per Spotify e Youtube utilizzando le credenziali caricate.

In [ ]:
client_id, client_secret, redirect_uri = load_spotify_config()

if not all([client_id, client_secret, redirect_uri]):
	print("Collega il tuo account Spotify.")

	client_id = input("\nInserisci il Client ID: ").strip() if not client_id else client_id
	client_secret = input("Inserisci il Client SECRET: ").strip() if not client_secret else client_secret
	redirect_uri = input("Inserisci il Redirect URI: ").strip() if not redirect_uri else redirect_uri
	print("\n")
else:
	print("✅ Credenziali Spotify caricate correttamente.\n")

# Crea il client Spotify.
sp = get_spotify_client(client_id, client_secret, redirect_uri)

# Crea il client Youtube.
yt_music = YTMusic()

if sp is None:
	print("❌ Impossibile creare il client Spotify. Verifica le credenziali inserite.")

## Inizializzazione sessione

Inizializza le strutture dati di sessione:
- `playlists`: dizionario delle playlist locali (nome e CSV associato)
- `found_tracks`: lista dei risultati di ricerca su Spotify
- `selected_tracks`: lista dei brani selezionati per la creazione di una nuova playlist

In [ ]:
playlists = {}
found_tracks = []
selected_tracks = []

print("✅ Stato di sessione inizializzato.")
print(f"Playlist locali: {len(playlists)}")

## Add playlist

1. Richiede l'URL di una playlist Spotify esistente.
2. Estrae l'ID playlist con `get_playlist_id_from_url()`.
3. Recupera i metadati della playlist con `sp.playlist()`.
4. Aggiunge la playlist al dizionario `playlists`.

In [ ]:
url = input("👉 Inserisci URL della playlist Spotify: ").strip()

playlist_id = get_playlist_id_from_url(url)

if not playlist_id:
	print("❌ Playlist ID non valido.\n")
else:
	playlist_info = sp.playlist(playlist_id)
	playlists[playlist_id] = {"name": get_safename(playlist_info['name'])}

	print(f"\n✅ Playlist aggiunta: {playlist_id} | {playlist_info['name']}")

## Print playlist

Stampa le playlist inserite nella sessione corrente tramite `print_local_playlists()`.

In [ ]:
if not playlists:
	print("⚠️ Nessuna playlist inserita.")
else:
	print("\n🎧 Playlist inserite:")
	playlist_ids = print_local_playlists(playlists)

## Process playlist

1. Selezione della playlist locale da elaborare.
2. Specifica della cartella di output.
3. Selezione del `QueryType` (tipo di ricerca YouTube).
4. Estrazione delle tracce con `extract_playlist_tracks()`.
5. Ricerca delle corrispondenze su YouTube con `search_from_youtube()`.
6. Esportazione dei risultati in CSV tramite `process_playlist()`.
7. Aggiornamento di `playlists` con il percorso del CSV generato.

In [ ]:
if not playlists:
	print("⚠️ Nessuna playlist da elaborare.")
else:

	print("\n🎧 Playlist inserite:")
	playlist_ids = print_local_playlists(playlists)
	playlist_scelta = input(f"\n👉 Seleziona un'opzione (1-{len(playlists)}): ").strip()

	if not playlist_scelta.isdigit() or not 1 <= int(playlist_scelta) <= len(playlists):
		print("❌ Opzione non valida.")
	else:

		playlist_scelta = playlist_ids[int(playlist_scelta) - 1]

		dir_scelta = input("👉 Inserisci la cartella di output (default: 'output'): ") \
			.strip() \
			.strip("\"") \
			.strip("\'") \
			or "output"

		print("👉 Inserisci il tipo di query:")
		for idx, query in enumerate(QUERY_LIST, 1):
			print(f"{idx}. {query}")
		query_scelta = input(f"\n👉 Seleziona un'opzione (1-{len(QUERY_LIST)}): ").strip()

		if not query_scelta.isdigit() or not 1 <= int(query_scelta) <= len(QUERY_LIST):
			print("❌ Opzione non valida.")
		else:

			query_scelta = int(query_scelta)
			playlist_info = playlists[playlist_scelta]

			if not playlist_info:
				print("❌ Playlist non valida.")
			else:

				csv_path = process_playlist(
					sp, yt_music,
					playlist_id=playlist_scelta,
					playlist_name=playlist_info["name"],
					query_type=QUERY_LIST[query_scelta - 1],
					start=0, length=None,
					output_dir=dir_scelta
				)

				playlists[playlist_scelta].update({"csv": csv_path})
				print(f"✅ File salvato in: '{csv_path}'")
				print("✅ Elaborazione completata.")

## Download playlist

1. Selezione del file CSV (da una playlist elaborata in sessione o inserendo manualmente il percorso).
2. Specifica della cartella di download.
3. Per ogni traccia, selezione interattiva del link YouTube da scaricare.
4. Avvio del download con `download_from_csv()`, gestendo sia audio (mp3) che video (mp4).

In [ ]:
if not playlists or all(not data.get("csv") for data in playlists.values()):

	print("⚠️ Nessun file CSV generato in questa sessione.")
	scelta_csv = input("👉 Inserisci il percorso del file CSV: ") \
		.strip() \
		.strip("\"") \
		.strip("\'")

	if not scelta_csv:
		print("❌ Nessun file fornito.\n")
		csv_file = None
	else:
		csv_file = scelta_csv
		playlist_name = Path(csv_file).stem

else:

	print("\n🎧 Playlist elaborate:")
	playlist_csv = get_processed_playlists(playlists)
	scelta_csv = input(f"\n👉 Seleziona un'opzione (1-{len(playlist_csv)}): ").strip()

	if not scelta_csv.isdigit() or not 1 <= int(scelta_csv) <= len(playlist_csv):
		print("❌ Opzione non valida.")
		csv_file = None
	else:
		_, csv_file, playlist_name = playlist_csv[int(scelta_csv) - 1]

if csv_file and not Path(csv_file).exists():
	print(f"❌ Il file '{csv_file}' non esiste.\n")
	csv_file = None

if csv_file:

	dir_scelta = input("👉 Inserisci la cartella di download (default: 'downloads'): ") \
		.strip() \
		.strip("\"") \
		.strip("\'") \
		or "downloads"

	try:
		print(f"\n🎵 Avvio download per '{playlist_name}'...")

		download_path = download_from_csv(
			csv_file,
			playlist_name,
			output_dir=dir_scelta
		)

		print(f"✅ Media salvati in: {download_path}")
		print("✅ Elaborazione completata.")

	except Exception as e:
		print(f"❌ Errore durante il download: {e}\n")